# Epic Game Pass When? - pipeline runner

The one-stop orchestrator. To refresh the app:
1. Drop new scrape dumps into `data/raw/` (same filenames as before).
2. Run the cells top to bottom (or **Run All**).
3. Review the training metrics and the git diff, then run the push cell to ship to **dev**.

Each stage just calls a module in `pipeline/` - the logic lives there, this notebook only orchestrates.

**Prereqs:** open this from the repo root; set the `RAWG_API_KEY` environment variable for enrichment. Production is never touched here - the push goes to the `dev` branch only (prod is a separate manual merge).

In [ ]:
from pipeline import config, ingest, enrich, train, deploy

print("Repo root:", config.REPO_ROOT)
print("RAWG_API_KEY set:", bool(config.RAWG_API_KEY))
config.ensure_dirs()

## 1. Ingest - raw dumps -> `data/processed/`
Parses the `data/raw/` scrape files into standardized `*_Processed.csv`.

In [ ]:
ingest.run()

## 2. Enrich - RAWG fill + merge -> `data/canonical/`
Fills missing publisher/developer/release/metacritic via RAWG (and the local cache), then merges into the canonical datasets. Needs `RAWG_API_KEY` for any rows not already in the cache. Backups land in `data/backups/`.

In [ ]:
enrich.run()

## 3. Train - `data/canonical/` -> `models/`
Trains one XGBoost model per platform. Review the MAE (days) and R2 below before shipping.

In [ ]:
import pandas as pd
metrics = train.run()
pd.DataFrame(metrics)

## 4. Deploy - sync `data/canonical/` + `models/` -> `apps/backend/`
Copies the canonical CSVs and trained artifacts into the backend so the API serves the new models.

In [ ]:
deploy.run()

## 5. Review, then ship to `dev`
Inspect the working tree first. Then run the push cell to commit and push to `dev`, which auto-deploys the dev environment. Production stays a separate manual merge (`dev -> main`).

In [ ]:
!git status --short

In [ ]:
# Uncomment to ship to dev (auto-deploys dev; prod is a separate manual merge):
# !git add -A && git commit -m "data refresh: retrain models + redeploy" && git push origin dev